# CRISP-DM - Etapa 05: Avaliação dos Resultados
## Tema 10: Indústria, Energia e Meio Ambiente (AI4I 2020 Predictive Maintenance)

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

# carrear base para recuperar os rotulos isolados
df_raw = pd.read_csv("../data/raw/ai4i2020.csv")
X = pd.read_parquet("../data/processed/ai4i2020_processed.parquet")

# treinar o K-Means final (k=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_raw["Cluster"] = kmeans.fit_predict(X)

print("Distribuição total de máquinas por Cluster:")
print(df_raw["Cluster"].value_counts())

### 1. Cruzamento dos Clusters com as Falhas Reais (Machine Failure)
Validação contra o gabarito isolado na Etapa 02 (Regra do PBL).

In [ ]:
# tabela cruzada entre Cluster e Falha Real
cruzamento = pd.crosstab(df_raw["Cluster"], df_raw["Machine failure"], margins=True)
cruzamento["% Falhas no Cluster"] = (cruzamento[1] / cruzamento["All"] * 100).round(2)

print("Tabela Cruzada: Cluster vs Machine Failure (0 = Sem Falha, 1 = Falha Real):")
cruzamento

### 2. Perfil Operacional de Cada Cluster (Personas de Máquinas)
Médias das variáveis operacionais em escala original para interpretação de negócio.

In [ ]:
# medias operacionais por cluster
colunas_sensores = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

perfil = df_raw.groupby("Cluster")[colunas_sensores].mean().round(2)
perfil["Total_Maquinas"] = df_raw["Cluster"].value_counts()
perfil["Total_Falhas"] = df_raw.groupby("Cluster")["Machine failure"].sum()
perfil["Taxa_Falha_%"] = (perfil["Total_Falhas"] / perfil["Total_Maquinas"] * 100).round(2)

perfil